In [134]:
import pandas as pd
import os
import geopandas as gpd
import pickle
from typing import List, Dict,Tuple,Set

# 1- Collect Site's Data

## 1.1- Collect Linking Tool Results

* a. **Define the Linking Tool Results Directory:** Assuming you have the following standard file structure:

    ```
    repository
        |___Bidirectional_Linking_Tool
            |___results
        |___BC-CLEWS-Model
            |__from_linking_tool
                |__results
    ```

    - `repository`: All Git Repositories' Root directory.
    - `Bidirectional_Linking_Tool`: Directory containing the _Bidirectional Linking Tool_.
        - `results`: Subdirectory within *Bidirectional_Linking_Tool*.
    - `BC-CLEWS-Model`: Directory containing the _BC-CLEWS-Model_ git repo.
        - `from_linking_tool`: Subdirectory within _BC-CLEWS-Model_.
            - `results`: Subdirectory within *from_linking_tool* , contains the pickle files from Linking tool.

In [135]:
linking_tool_directs={
    'root': 'Bidirectional_Linking_Tool',
    'results': 'results',
}

bc_nexus_direct={
    'root': 'BC-CLEWS-Model',
    'collect_linking_tool_results':'from_linking_tool',
    'results': 'results',
}

print(f"Please make sure you have '{linking_tool_directs['root']}' directory present in your 'repository' root.")

directories = 'new_sites_timeslice','new_sites'

for directory in directories:
    # Check if the directory exists, and create it if it doesn't
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory '{directory}' created successfully.")
    else:
        print(f"Directory '{directory}' already exists.")

Please make sure you have 'Bidirectional_Linking_Tool' directory present in your 'repository' root.
Directory 'new_sites_timeslice' already exists.
Directory 'new_sites' already exists.


* b. Copy datafiles FROM > TO defined directories with Bash Cmd

__!!! Caution__: The Bash cmd execution relies on the aforemention __standard Folder Structure__

> copy-paste result pickle files (__cluster info__ & __timeseries of clusters__) from Linking tool to BC-Nexus directories for further analysis.

In [136]:
!cd .. && cd .. && mkdir -p {bc_nexus_direct['root']}/{bc_nexus_direct['collect_linking_tool_results']}/{bc_nexus_direct['results']} && cp {linking_tool_directs['root']}/{linking_tool_directs['results']}/*.pkl {bc_nexus_direct['root']}/{bc_nexus_direct['collect_linking_tool_results']}/{bc_nexus_direct['results']}

# 2- Load related SETS Defined for BCNexus
> The timeslices are defined via __BCNexus Config file__.

## 2.1- Function to load the associated SETS from the BCNexus input datafiles.

In [137]:
def get_sets_list(
        sets_required: list[str],
        sets_datafile_directory: str) -> dict[str, list]:
    
    dataframes = {}
    sets_lists = {}

    # Iterate through the sets and read the CSV files
    for set_name in sets_required:
        file_path = os.path.join(sets_datafile_directory, f"{set_name}.csv")
        dataframes[set_name] = pd.read_csv(file_path)
        sets_lists[set_name] = dataframes[set_name]['VALUE'].to_list()

    return sets_lists



* > Usage of Function(2.1)

In [138]:
sets_required = ['TIMESLICE', 'TECHNOLOGY','YEAR','REGION']
sets_datafile_directory = '/localhome/mei3/eliasinul/repositories/BC-CLEWS-Model/datapackage/SETS/CLEWsy_outputs/' #later to be configured via config file

sets_lists = get_sets_list(sets_required, sets_datafile_directory)

#Load the lists out of the dictioary for further analysis
TIMESLICES:list[str] = sets_lists['TIMESLICE']
seen_TECHNOLOGIES:list[str] = sets_lists['TECHNOLOGY']
YEARS:list[int]=sets_lists['YEAR']
REGION:list[str]=sets_lists['REGION']

# 3- Define Season and Timeslice Configuration

> Currently hardcoded, but ideally to be replaced by __TSA (Time Slice Aggregation)__ Algorithm or TS downscaling code

In [139]:
season_config = {
    "Winter1": {"start": "2021-01-01", "end": "2021-03-19", "daytime": {"day": 8, "night": 17}},
    "Spring": {"start": "2021-03-20", "end": "2021-06-19", "daytime": {"day": 6, "night": 20}},
    "Summer": {"start": "2021-06-20", "end": "2021-09-21", "daytime": {"day": 6, "night": 21}},
    "Fall": {"start": "2021-09-22", "end": "2021-12-20", "daytime": {"day": 8, "night": 18}},
    "Winter2": {"start": "2021-12-21", "end": "2021-12-31", "daytime": {"day": 8, "night": 17}}
}

# Mapping dictionary
timeslice_name_mapping = {
    'Fall_D': 'SEA2D',
    'Fall_N': 'SEA2N',
    'Out_of_Season': 'NA',
    'Spring_D': 'SEA3D',
    'Spring_N': 'SEA3N',
    'Summer_D': 'SEA4D',
    'Summer_N': 'SEA4N',
    'Winter1_D': 'SEA1D',
    'Winter1_N': 'SEA1N',
    'Winter2_D': 'SEA1D',
    'Winter2_N': 'SEA1N'
}

# 4- Downscale the Timeseries collected from Linking Tool

## 4.1- Function to Convert and Extract Timeslice Downscaling Results

> to be automated further inline with timeslice configuration for __BC-Nexus__

In [140]:
# def create_timeslice_from_cluster(cluster_ts_pickle,site_name,season_config,timeslice_name_mapping):

def create_timeslice_from_cluster(
    cluster_ts_pickle: str, 
    site_name: str, 
    season_config: Dict[str, Dict[str, Dict[str, int]]], 
    timeslice_name_mapping: Dict[str, str]
    ) -> pd.DataFrame:

    df_ts=pd.read_pickle(cluster_ts_pickle)

    # Convert season_config dates to datetime with error handling
    for season, dates in season_config.items():
        try:
            dates["start"] = pd.to_datetime(dates["start"])
            dates["end"] = pd.to_datetime(dates["end"])
        except ValueError as e:
            print(f"Error converting dates for season '{season}': {e}")

    # Improved function for clarity and error handling
    def get_season_time_of_day(datetime):
        for season, dates in season_config.items():
            if dates["start"] <= datetime <= dates["end"]:
                # Handle specific time ranges for day and night
                if 0 <= datetime.hour <= dates["daytime"]["day"]:
                    return f"{season}_N"  # Night
                elif dates["daytime"]["day"] <= datetime.hour <= dates["daytime"]["night"]:
                    return f"{season}_D"  # Day
                else:  # Outside daytime range
                    return f"{season}_N"  # Assuming night for values outside day/night times
        # Handle date outside any season (optional: return specific value or raise an error)
        return "Out_of_Season"  # Example handling

    # Create the 'season_daytime' column with error handling
    try:
        df_ts['season_daytime'] = df_ts.index.map(get_season_time_of_day)
    except TypeError as e:
        print("Error applying function: ", e)

    # Display the DataFrame
    # Group by season and day/night
    grouped_df = df_ts.groupby(['season_daytime']).agg({
        site_name: ['mean']  # You can add more aggregation functions here e.g 'std'
    })

    # Display the grouped DataFrame
    grouped_df['CF_mean'] = grouped_df[site_name]['mean']
    grouped_df['TIMESLICE'] = grouped_df.index.map(timeslice_name_mapping)
    grouped_df=grouped_df.groupby('TIMESLICE').mean() 
    
    grouped_df.columns=grouped_df.columns.get_level_values(0)
    return grouped_df,df_ts

In [141]:
# def create_n_save_timeslice_files(ts_pickle,cluster_pickle,asset_type,season_config,timeslice_name_mapping):

def create_n_save_timeslice_files(
    ts_pickle: str, 
    cluster_pickle: str, 
    asset_type: str, 
    season_config: Dict[str, Dict[str, Dict[str, int]]], 
    timeslice_name_mapping: Dict[str, str]
    ) -> Tuple[Dict[str, pd.DataFrame], Dict[str, pd.DataFrame], gpd.GeoDataFrame]:
  
    sites_df=gpd.GeoDataFrame(pd.read_pickle(cluster_pickle))
    sites_without_geom=sites_df.drop(columns=['geometry'])
    sites_without_geom.to_csv(f'new_sites/{asset_type}.csv')
    
    sites=list(sites_df.index)
    cluster_ts_timeslice = {}
    cluster_ts={}

    for site in sites:
        cluster_ts_timeslice[site],cluster_ts = create_timeslice_from_cluster(ts_pickle, site,season_config,timeslice_name_mapping)
        cluster_ts_timeslice[site].to_csv(f'new_sites_timeslice/{asset_type}_{site}.csv')
        
    
    print(f"BCNexus compatible timeseries created for - {asset_type} Clusters")
    return cluster_ts,cluster_ts_timeslice,sites_df

* Use of the timeslice conversion function
> tailored for hardcoded season config. Later to be replaced by standard seasonal config or time series aggregation code

In [142]:
solar_ts,solar_timeslices,solar_sites=create_n_save_timeslice_files('results/Solar_Top_Sites_Clustered_CF_timeseries.pkl','results/Solar_Top_Sites_Clustered.pkl','Solar',season_config,timeslice_name_mapping)
wind_ts,wind_timeslices,wind_sites=create_n_save_timeslice_files('results/Wind_Top_Sites_Clustered_CF_timeseries.pkl','results/Wind_Top_Sites_Clustered.pkl','Wind',season_config,timeslice_name_mapping)

BCNexus compatible timeseries created for - Solar Clusters
BCNexus compatible timeseries created for - Wind Clusters


# 5- Impute additional Datafields associated to the Nexus compatible timeslices

### 5.1- Define the list of years and planning horizon
> Now hardcoded but later to be configured via config file

## 5.2- Process the Results and plug-in other required Datafields

> Hardcoded for 1 region only

In [143]:
def impute_additional_datafields_to_timeslices_forNexus(
        downscaled_timeslice_dict: Dict[str, pd.DataFrame],
        TIMESLICES: List[str],
        seen_TECHNOLOGIES: Set[str],
        technology_prefix: str,
        site_no_start: int,
        sites_COD: int,
        YEARS:List[int],
        REGIONS:List[str]
    ) -> Dict[str, pd.DataFrame]:
    """
    Processes timeslice data for Nexus compatibility.

    Parameters:
    __
    downscaled_timeslice_dict (dict): Dictionary of downscaled timeslice dataframes.
    TIMESLICES (list): List of valid timeslices.
    seen_TECHNOLOGIES (set): Set of seen technology names to check for duplicates.
    technology_prefix (str): Prefix for the technology names.
    site_no_start (int): Starting number for the site numbering.
    sites_COD (int): Start year for the sites.
    planning_horizon (int): End year for the planning horizon.
    region (str): Region name to be added to the data.

    Returns:
    __
    dict: Dictionary of processed dataframes for Nexus data intake validation.
    """

    _nexus_timeslices_ = {}
    planning_horizon=max(YEARS)
    years = list(range(sites_COD, planning_horizon + 1))
    operational_years_count = planning_horizon - sites_COD + 1

    # Process each site in the timeslice dictionary
    for site_sl, (site, site_ts_df) in enumerate(downscaled_timeslice_dict.items(), start=1):
        # Reset the index to turn 'timeslices' into a regular column
        site_ts_df=site_ts_df.reset_index()
        # Select only the 'TIMESLICE' and 'CF_mean' columns
        site_ts_df = site_ts_df[['TIMESLICE', 'CF_mean']]

        # Filter data based on TIMESLICES
        _clean_data_ = site_ts_df[site_ts_df['TIMESLICE'].isin(TIMESLICES)]
        _clean_data_ = _clean_data_.rename(columns={'CF_mean': 'VALUE'})

        # Repeat each row for each year in the operational period
        _all_datafields_ = pd.concat([_clean_data_] * operational_years_count, ignore_index=True)
        _all_datafields_['YEAR'] = [year for year in years for _ in range(len(_clean_data_))]

        # Add 'REGION' and 'TECHNOLOGY' columns with constant values
        _all_datafields_['REGION'] = REGIONS[0] # Hardcoded for 1 region

        technology_name = f"{technology_prefix}{(site_sl + site_no_start - 1):02d}"
        _all_datafields_['TECHNOLOGY'] = technology_name

        # Check if the technology already exists
        if technology_name in seen_TECHNOLOGIES:
            print(f"Technology {technology_name} already exists. Recommended to revise the Site Start no. of the TECHNOLOGY.")
            continue
        else:
            print(f">>> Technology {technology_name} Created !\n *** Recommended to check the BCNexus Config file for associated datafields for technology >> '{technology_name}'")
            seen_TECHNOLOGIES.append(technology_name)  # should I save this to SETs ???

        # Assign the processed data to the nexus_timeslices dictionary
        _nexus_timeslices_[site] = _all_datafields_

    return _nexus_timeslices_

In [144]:
solar_sites_COD:int=2025 # Later to be configured via config file
nexus_timeslices_SOLAR=impute_additional_datafields_to_timeslices_forNexus(solar_timeslices,TIMESLICES,seen_TECHNOLOGIES,"PWRSOLB",3,solar_sites_COD,YEARS,REGION)

>>> Technology PWRSOLB03 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRSOLB03'


In [145]:
wind_sites_COD:int=2025 # Later to be configured via config file
nexus_timeslices_WIND=impute_additional_datafields_to_timeslices_forNexus(wind_timeslices,TIMESLICES,seen_TECHNOLOGIES,"PWRWNDB",13,solar_sites_COD,YEARS,REGION)

>>> Technology PWRWNDB13 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRWNDB13'
>>> Technology PWRWNDB14 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRWNDB14'
>>> Technology PWRWNDB15 Created !
 *** Recommended to check the BCNexus Config file for associated datafields for technology >> 'PWRWNDB15'


## 5.3- Save the datafiles to local drives

In [147]:
with open('new_sites_timeslice/solar_ts.pkl', 'wb') as file:
    pickle.dump(nexus_timeslices_SOLAR, file)

with open('new_sites_timeslice/wind_ts.pkl', 'wb') as file:
    pickle.dump(nexus_timeslices_WIND, file)